In [1]:
# Install the lightweight manager first
%pip install webdriver-manager --quiet

# Install selenium (the core tool)
%pip install selenium --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Install pandas (usually the one that takes longest due to size)
%pip install pandas --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.maximize_window()

def get_bulk_movies(target=5100):
    url = "https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31"
    driver.get(url)
    
    while True:
        try:
            # 1. Scroll down to make the button visible
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            
            # 2. Check current count
            current_elements = driver.find_elements(By.CLASS_NAME, "ipc-metadata-list-summary-item")
            print(f"Current movies loaded: {len(current_elements)}")
            
            if len(current_elements) >= target:
                print("Target reached!")
                break
            
            # 3. Targeted Click for the "50 more" button
            # We use XPath to find the span that contains the text '50 more'
            load_more_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), '50 more')]"))
            )
            
            # Use JavaScript click to bypass any overlay issues
            driver.execute_script("arguments[0].click();", load_more_button)
            
            # Wait for the new items to load
            time.sleep(3) 
            
        except Exception as e:
            print(f"No more '50 more' button found or error: {e}")
            break

    # 4. Final Data Extraction
    print("Beginning Data Extraction...")
    movie_data = []
    items = driver.find_elements(By.CLASS_NAME, "ipc-metadata-list-summary-item")
    
    for item in items:
        try:
            name = item.find_element(By.CLASS_NAME, "ipc-title__text").text
            # Remove leading numbers (e.g., "1. Venom")
            name = name.split('. ', 1)[-1] if '. ' in name else name
            
            story = item.find_element(By.CLASS_NAME, "ipc-html-content-inner-div").text
            
            movie_data.append({"Movie Name": name, "Storyline": story})
        except:
            continue
            
    df = pd.DataFrame(movie_data)
    df.to_csv("imdb_movies_2024.csv", index=False)
    print(f"Finished! Saved {len(df)} movies.")

get_bulk_movies(5100)
driver.quit()

Current movies loaded: 50
Current movies loaded: 100
Current movies loaded: 150
Current movies loaded: 200
Current movies loaded: 250
Current movies loaded: 300
Current movies loaded: 350
Current movies loaded: 400
Current movies loaded: 450
Current movies loaded: 500
Current movies loaded: 550
Current movies loaded: 600
Current movies loaded: 650
Current movies loaded: 700
Current movies loaded: 750
Current movies loaded: 800
Current movies loaded: 850
Current movies loaded: 900
Current movies loaded: 950
Current movies loaded: 1000
Current movies loaded: 1050
Current movies loaded: 1100
Current movies loaded: 1150
Current movies loaded: 1200
Current movies loaded: 1250
Current movies loaded: 1300
Current movies loaded: 1350
Current movies loaded: 1400
Current movies loaded: 1450
Current movies loaded: 1500
Current movies loaded: 1550
Current movies loaded: 1600
Current movies loaded: 1650
Current movies loaded: 1700
Current movies loaded: 1750
Current movies loaded: 1800
Current movi

In [3]:
%pip install nltk scikit-learn

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 17.0 MB/s  0:00:00

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [regex]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Download necessary NLTK data packs
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
print("All NLTK resource packages downloaded successfully!")

def clean_storyline(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove punctuation and numbers using Regex
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 3. Tokenization (Split into individual words)
    words = word_tokenize(text)
    
    # 4. Remove Stopwords (words like 'the', 'a', 'in' that add no unique context)
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    
    # 5. Lemmatization (Convert words to their base/root form, e.g., 'running' -> 'run')
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(words)

[nltk_data] Downloading package punkt to C:\Users\Praveen
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Praveen
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


All NLTK resource packages downloaded successfully!


[nltk_data] Downloading package stopwords to C:\Users\Praveen
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Praveen
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [8]:
def build_recommendation_engine(csv_path):
    # Load your scraped data
    df = pd.read_csv(csv_path)
    
    # Drop rows with empty titles or storylines
    df = df.dropna(subset=['Movie Name', 'Storyline'])
    
    print("Cleaning storylines...")
    df['Cleaned_Storyline'] = df['Storyline'].apply(clean_storyline)
    
    print("Vectorizing data with TF-IDF...")
    # Initialize TF-IDF Vectorizer
    tfidf = TfidfVectorizer(max_features=5000) # Keep top 5000 most important words
    
    # Fit and transform the cleaned storylines into a matrix
    tfidf_matrix = tfidf.fit_transform(df['Cleaned_Storyline'])
    
    # Save the processed data and models so your Streamlit app can load them instantly
    df.to_csv("processed_movies.csv", index=False)
    
    with open("tfidf_vectorizer.pkl", "wb") as f:
        pickle.dump(tfidf, f)
        
    with open("tfidf_matrix.pkl", "wb") as f:
        pickle.dump(tfidf_matrix, f)
        
    print("Preprocessing complete! Engine files generated successfully.")

# Run the builder function
# 
build_recommendation_engine("imdb_movies_2024.csv")

Cleaning storylines...
Vectorizing data with TF-IDF...
Preprocessing complete! Engine files generated successfully.


**COSINE SIMILARITY**

Cosine Similarity is a mathematical metric used to measure how similar two items are, regardless of their size.

In [12]:
def get_recommendations(user_input_storyline, top_n=5):
    # Load assets
    df = pd.read_csv("processed_movies.csv")
    with open("tfidf_vectorizer.pkl", "rb") as f:
        tfidf = pickle.load(f)
    with open("tfidf_matrix.pkl", "rb") as f:
        tfidf_matrix = pickle.load(f)
        
    # 1. Clean user input just like the dataset
    cleaned_input = clean_storyline(user_input_storyline)
    
    # 2. Transform user input into the TF-IDF space
    user_vector = tfidf.transform([cleaned_input])
    
    # 3. Compute Cosine Similarity between user input and all movies
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()
    
    # 4. Get the indices of the top N highest scores
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    
    # 5. Display results
    print(f"\n--- Top {top_n} Movie Recommendations ---")
    for rank, idx in enumerate(top_indices, 1):
        print(f"{rank}. {df.iloc[idx]['Movie Name']} (Match Score: {similarity_scores[idx]:.2f})")
        print(f"   Plot: {df.iloc[idx]['Storyline']}\n")

# Test example
test_plot = "A thrilling space mission where astronauts get lost in a black hole."
get_recommendations(test_plot)


--- Top 5 Movie Recommendations ---
1. Tales Beyond the Galaxy (Match Score: 0.39)
   Plot: Astronauts discover alarming evidence on a vacant planet. When catastrophe strikes, one searches for lost crew. A woman faces her history in space. Failed systems threaten a habitat. Two lost astronauts drift, seeking their vessel.

2. Beyond the Unknown (Match Score: 0.25)
   Plot: In this epic anthology drama and sci-fi story, an Astronaut gets stranded in space and his memory unlocks what he did in his past which transforms into the past, present, and future, also a woman discovers the truth about her husband.

3. Uranus 2324 (Match Score: 0.24)
   Plot: Bonded in their youth and separated by tragedy, an astronaut and a free diver bend the laws of space and time in their efforts to be together again.

4. Slingshot (Match Score: 0.20)
   Plot: An astronaut struggles to maintain his grip on reality aboard a possibly fatally compromised mission to Saturn's moon, Titan.

5. Alien: Romulus (Match